# 01 — Data Understanding

This notebook introduces the raw Olist e-commerce dataset and establishes the first analytical foundation for the SupplyGuard project.

SupplyGuard aims to simulate a real consulting project for an e-commerce marketplace: understanding delivery performance, identifying late delivery patterns, and later building a predictive model to estimate delivery delay risk before it affects the customer.

Before any cleaning, SQL modeling, EDA, or machine learning work, this notebook focuses on understanding the structure of the available data. The Olist dataset is relational, meaning that the information is distributed across several connected tables: orders, customers, sellers, products, order items, payments, reviews, geolocation, and category translations.

The goal of this notebook is to answer:

- What tables are available?
- What does each table represent?
- How many rows and columns does each table contain?
- Which columns contain missing values?
- Which columns look like dates or timestamps?
- What are the likely primary keys and foreign key relationships?
- What potential data quality or leakage risks should be considered later?

This notebook does not clean, transform, or model the data. Instead, it creates a clear baseline understanding of the raw data so that the next stages of the project can be built with better decisions and fewer assumptions.

Important leakage note: the future target will be based on whether an order was delivered after its estimated delivery date. However, variables known only after delivery must not be used as predictive features in the final machine learning model.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.api.types import is_object_dtype, is_string_dtype

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Project Paths

The project uses `pathlib` to define robust file paths. This makes the notebook easier to run from different working directories, such as the project root or the `notebooks/` folder.

In [2]:
current_path = Path.cwd().resolve()
possible_roots = [current_path, *current_path.parents]

PROJECT_ROOT = next((path for path in possible_roots if (path / "data" / "raw").exists()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root could not be detected. Expected to find a 'data/raw' folder.")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Raw data directory exists: {RAW_DATA_DIR.exists()}")

Project root: C:\Users\johan\Desktop\supplyguard-delivery-risk
Raw data directory: C:\Users\johan\Desktop\supplyguard-delivery-risk\data\raw
Raw data directory exists: True


## 2. Expected Raw CSV Files

The Olist dataset should contain 9 CSV files. Non-data files such as `.gitkeep` are ignored because only CSV files are relevant for this stage.

In [3]:
expected_csv_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

In [4]:
detected_csv_files = sorted([path.name for path in RAW_DATA_DIR.glob("*.csv")])
expected_file_names = sorted(expected_csv_files.values())

missing_files = sorted(set(expected_file_names) - set(detected_csv_files))
unexpected_csv_files = sorted(set(detected_csv_files) - set(expected_file_names))

file_check_df = pd.DataFrame({
    "table_name": list(expected_csv_files.keys()),
    "expected_file": list(expected_csv_files.values()),
    "file_found": [file_name in detected_csv_files for file_name in expected_csv_files.values()]
})

display(file_check_df)

print(f"Detected CSV files: {len(detected_csv_files)}")
print(f"Expected CSV files: {len(expected_file_names)}")
print(f"Missing files: {missing_files if missing_files else 'None'}")
print(f"Unexpected CSV files: {unexpected_csv_files if unexpected_csv_files else 'None'}")

,table_name,expected_file,file_found
0,customers,olist_customers_dataset.csv,True
1,geolocation,olist_geolocation_dataset.csv,True
2,order_items,olist_order_items_dataset.csv,True
3,order_payments,olist_order_payments_dataset.csv,True
4,order_reviews,olist_order_reviews_dataset.csv,True
5,orders,olist_orders_dataset.csv,True
6,products,olist_products_dataset.csv,True
7,sellers,olist_sellers_dataset.csv,True
8,category_translation,product_category_name_translation.csv,True


Detected CSV files: 9
Expected CSV files: 9
Missing files: None
Unexpected CSV files: None


In [5]:
if missing_files:
    raise FileNotFoundError(f"The following expected files are missing from data/raw: {missing_files}")

print("All expected CSV files were found successfully.")

All expected CSV files were found successfully.


## 3. Load Raw DataFrames

Each CSV file is loaded into a dictionary of DataFrames. The dictionary keys use short, readable table names to make the rest of the notebook easier to work with.

In [6]:
dfs = {}

for table_name, file_name in expected_csv_files.items():
    file_path = RAW_DATA_DIR / file_name
    dfs[table_name] = pd.read_csv(file_path)

print(f"Loaded tables: {len(dfs)}")
print(list(dfs.keys()))

Loaded tables: 9
['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation']


In [7]:
shape_summary = pd.DataFrame([
    {"table_name": table_name, "rows": df.shape[0], "columns": df.shape[1]}
    for table_name, df in dfs.items()
]).sort_values("rows", ascending=False)

display(shape_summary)

,table_name,rows,columns
1,geolocation,1000163,5
2,order_items,112650,7
3,order_payments,103886,5
0,customers,99441,5
5,orders,99441,8
4,order_reviews,99224,7
6,products,32951,9
7,sellers,3095,4
8,category_translation,71,2


In [8]:
selected_table = "orders"

print(f"Table: {selected_table}")
print(f"Shape: {dfs[selected_table].shape}")

display(dfs[selected_table].head())

Table: orders
Shape: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 4. Table-Level Overview and Initial Data Quality Summary

Before cleaning or transforming the data, this section creates a high-level overview of every raw table.

The objective is to understand the size, structure, missingness, duplicate rows, memory usage, column types, and possible datetime-like fields in each dataset.

At this stage, no data is modified. This notebook focuses on data understanding only.

In [9]:
def detect_datetime_like_columns(df, sample_size=250, parse_threshold=0.80):
    """
    Detects columns that are likely to represent dates or timestamps.

    Detection is based on:
    1. Column name keywords.
    2. A lightweight parsing check on a small non-null sample.

    This function does not modify the DataFrame.
    """
    datetime_keywords = ("date", "time", "timestamp", "datetime")
    datetime_like_columns = []

    for column in df.columns:
        column_lower = column.lower()
        name_suggests_datetime = any(keyword in column_lower for keyword in datetime_keywords)

        parse_suggests_datetime = False

        if df[column].dtype == "object":
            sample = df[column].dropna().astype(str).head(sample_size)

            if len(sample) > 0:
                parsed_sample = pd.to_datetime(sample, errors="coerce")
                parse_suggests_datetime = parsed_sample.notna().mean() >= parse_threshold

        if name_suggests_datetime or parse_suggests_datetime:
            datetime_like_columns.append(column)

    return datetime_like_columns

In [10]:
table_summary_records = []

for table_name, df in dfs.items():
    rows, columns = df.shape
    total_cells = rows * columns
    total_missing_values = df.isna().sum().sum()
    missing_percentage = total_missing_values / total_cells * 100 if total_cells > 0 else 0

    table_summary_records.append({
        "table_name": table_name,
        "rows": rows,
        "columns": columns,
        "duplicate_rows": df.duplicated().sum(),
        "total_missing_values": total_missing_values,
        "missing_percentage": missing_percentage,
        "memory_usage_mb": df.memory_usage(deep=True).sum() / 1024**2,
        "numeric_columns": df.select_dtypes(include="number").shape[1],
        "object_columns": df.select_dtypes(include=["object", "string"]).shape[1],
        "datetime_like_columns": len(detect_datetime_like_columns(df))
    })

table_summary = (
    pd.DataFrame(table_summary_records)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

display(table_summary)

,table_name,rows,columns,duplicate_rows,total_missing_values,missing_percentage,memory_usage_mb,numeric_columns,object_columns,datetime_like_columns
0,geolocation,1000163,5,261831,0,0.00,50.12,3,2,0
1,order_items,112650,7,0,0,0.00,18.37,3,4,1
2,order_payments,103886,5,0,0,0.00,8.11,3,2,0
3,customers,99441,5,0,0,0.00,11.03,1,4,0
4,orders,99441,8,0,4908,0.62,21.95,0,8,4
5,order_reviews,99224,7,0,145903,21.01,17.84,1,6,2
6,products,32951,9,0,2448,0.83,3.73,7,2,0
7,sellers,3095,4,0,0,0.00,0.22,1,3,0
8,category_translation,71,2,0,0,0.00,0.00,0,2,0


### Initial Observations

This summary provides the first overview of the raw data structure.

Key aspects to check at this stage:

- Which tables contain the most records.
- Whether any table has complete duplicate rows.
- Which tables contain missing values.
- Which tables are mostly numeric or mostly categorical/text-based.
- Which columns may need datetime conversion in a later cleaning phase.

No cleaning decisions are applied here. Any potential issues identified in this section will be handled later in the cleaning notebook.

In [30]:
datetime_like_summary = pd.DataFrame([
    {
        "table_name": table_name,
        "datetime_like_columns": detect_datetime_like_columns(df)
    }
    for table_name, df in dfs.items()
])

with pd.option_context("display.max_colwidth", None):
    display(datetime_like_summary)

,table_name,datetime_like_columns
0,customers,[]
1,geolocation,[]
2,order_items,[shipping_limit_date]
3,order_payments,[]
4,order_reviews,"[review_creation_date, review_answer_timestamp]"
5,orders,"[order_purchase_timestamp, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date]"
6,products,[]
7,sellers,[]
8,category_translation,[]


## 5. Column Inventory

The following inventory lists the columns available in each raw table. This helps document the structure of the dataset and supports the identification of relationships between tables.

In [12]:
column_inventory = pd.DataFrame([
    {
        "table_name": table_name,
        "column_position": position,
        "column_name": column
    }
    for table_name, df in dfs.items()
    for position, column in enumerate(df.columns, start=1)
])

#display(column_inventory)

In [29]:
column_list_summary = pd.DataFrame([
    {
        "table_name": table_name,
        "column_count": len(df.columns),
        "columns": ", ".join(df.columns)
    }
    for table_name, df in dfs.items()
])

with pd.option_context("display.max_colwidth", None):
    display(column_list_summary)

,table_name,column_count,columns
0,customers,5,"customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state"
1,geolocation,5,"geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state"
2,order_items,7,"order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value"
3,order_payments,5,"order_id, payment_sequential, payment_type, payment_installments, payment_value"
4,order_reviews,7,"review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp"
5,orders,8,"order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date"
6,products,9,"product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm"
7,sellers,4,"seller_id, seller_zip_code_prefix, seller_city, seller_state"
8,category_translation,2,"product_category_name, product_category_name_english"


## 6. Data Types by Table

This section reviews the data types automatically assigned by pandas when loading the raw CSV files.

The objective is to understand the structure of each table without modifying the data. At this stage, columns that represent dates or timestamps are expected to appear mostly as `object`, because datetime conversion will be handled later during the cleaning phase.

To keep the notebook concise, the full column-level dtype summary is created and stored, but only a compact overview is displayed.

In [14]:
dtype_summary = pd.DataFrame([
    {
        "table_name": table_name,
        "column_name": column,
        "dtype": str(df[column].dtype),
        "non_null_values": df[column].notna().sum(),
        "missing_values": df[column].isna().sum(),
        "missing_percentage": df[column].isna().mean() * 100,
        "unique_values": df[column].nunique(dropna=True)
    }
    for table_name, df in dfs.items()
    for column in df.columns
])

In [15]:
dtype_overview = (
    dtype_summary
    .groupby(["table_name", "dtype"])
    .size()
    .reset_index(name="column_count")
    .sort_values(["table_name", "dtype"])
)

display(dtype_overview)

,table_name,dtype,column_count
0,category_translation,str,2
1,customers,int64,1
2,customers,str,4
3,geolocation,float64,2
4,geolocation,int64,1
5,geolocation,str,2
6,order_items,float64,2
7,order_items,int64,1
8,order_items,str,4
9,order_payments,float64,1


In [16]:
selected_table = "orders"

selected_dtype_summary = (
    dtype_summary[dtype_summary["table_name"] == selected_table]
    .sort_values("column_name")
    .reset_index(drop=True)
)

display(selected_dtype_summary)

,table_name,column_name,dtype,non_null_values,missing_values,missing_percentage,unique_values
0,orders,customer_id,str,99441,0,0.00,99441
1,orders,order_approved_at,str,99281,160,0.16,90733
2,orders,order_delivered_carrier_date,str,97658,1783,1.79,81018
3,orders,order_delivered_customer_date,str,96476,2965,2.98,95664
4,orders,order_estimated_delivery_date,str,99441,0,0.00,459
5,orders,order_id,str,99441,0,0.00,99441
6,orders,order_purchase_timestamp,str,99441,0,0.00,98875
7,orders,order_status,str,99441,0,0.00,8


The detailed column-level dtype information is available in the `dtype_summary` DataFrame.

For focused inspection, the `selected_table` variable can be changed to any table name in the `dfs` dictionary.

## 7. Missing Values by Column

This section identifies missing values at column level.

Only columns with missing values are displayed, making the output easier to review. Missing values are not imputed or removed in this notebook.

In [17]:
def missing_values_by_table(df, table_name):
    missing_df = (
        df.isna()
        .sum()
        .reset_index()
        .rename(columns={"index": "column_name", 0: "missing_values"})
    )

    missing_df["table_name"] = table_name
    missing_df["missing_percentage"] = missing_df["missing_values"] / len(df) * 100
    missing_df = missing_df[missing_df["missing_values"] > 0]

    return missing_df[["table_name", "column_name", "missing_values", "missing_percentage"]].sort_values(
        "missing_percentage", ascending=False
    )

In [18]:
missing_values_summary = pd.concat(
    [missing_values_by_table(df, table_name) for table_name, df in dfs.items()],
    ignore_index=True
)

display(missing_values_summary)

,table_name,column_name,missing_values,missing_percentage
0,order_reviews,review_comment_title,87656,88.34
1,order_reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,orders,order_delivered_carrier_date,1783,1.79
4,orders,order_approved_at,160,0.16
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01


## 8. Preliminary Key Analysis

This section performs a preliminary inspection of potential primary keys and foreign keys.

The objective is not to enforce database constraints yet, but to understand how the raw Olist tables may relate to each other.

Some tables are expected to have simple unique identifiers, while others may require composite keys. For example, order items and order payments can contain multiple rows per order.

In [19]:
key_candidates = {
    "customers": {
        "primary_key_candidates": [["customer_id"]],
        "foreign_key_candidates": []
    },
    "geolocation": {
        "primary_key_candidates": [],
        "foreign_key_candidates": []
    },
    "order_items": {
        "primary_key_candidates": [["order_id", "order_item_id"]],
        "foreign_key_candidates": [["order_id"], ["product_id"], ["seller_id"]]
    },
    "order_payments": {
        "primary_key_candidates": [["order_id", "payment_sequential"]],
        "foreign_key_candidates": [["order_id"]]
    },
    "order_reviews": {
        "primary_key_candidates": [["review_id"]],
        "foreign_key_candidates": [["order_id"]]
    },
    "orders": {
        "primary_key_candidates": [["order_id"]],
        "foreign_key_candidates": [["customer_id"]]
    },
    "products": {
        "primary_key_candidates": [["product_id"]],
        "foreign_key_candidates": [["product_category_name"]]
    },
    "sellers": {
        "primary_key_candidates": [["seller_id"]],
        "foreign_key_candidates": []
    },
    "category_translation": {
        "primary_key_candidates": [["product_category_name"]],
        "foreign_key_candidates": []
    }
}

In [20]:
def evaluate_key_candidate(df, columns):
    total_rows = len(df)

    if not set(columns).issubset(df.columns):
        return {
            "columns_exist": False,
            "non_null_rows": np.nan,
            "unique_combinations": np.nan,
            "is_unique": False,
            "has_missing_values": np.nan
        }

    key_data = df[columns]
    non_null_rows = key_data.dropna().shape[0]
    unique_combinations = key_data.drop_duplicates().shape[0]

    return {
        "columns_exist": True,
        "non_null_rows": non_null_rows,
        "unique_combinations": unique_combinations,
        "is_unique": unique_combinations == total_rows,
        "has_missing_values": non_null_rows < total_rows
    }

In [21]:
primary_key_records = []

for table_name, candidates in key_candidates.items():
    df = dfs[table_name]

    for columns in candidates["primary_key_candidates"]:
        result = evaluate_key_candidate(df, columns)

        primary_key_records.append({
            "table_name": table_name,
            "candidate_key": " + ".join(columns),
            "rows": len(df),
            **result
        })

primary_key_summary = pd.DataFrame(primary_key_records)

display(primary_key_summary)

,table_name,candidate_key,rows,columns_exist,non_null_rows,unique_combinations,is_unique,has_missing_values
0,customers,customer_id,99441,True,99441,99441,True,False
1,order_items,order_id + order_item_id,112650,True,112650,112650,True,False
2,order_payments,order_id + payment_sequential,103886,True,103886,103886,True,False
3,order_reviews,review_id,99224,True,99224,98410,False,False
4,orders,order_id,99441,True,99441,99441,True,False
5,products,product_id,32951,True,32951,32951,True,False
6,sellers,seller_id,3095,True,3095,3095,True,False
7,category_translation,product_category_name,71,True,71,71,True,False


### Preliminary Primary Key Notes

A valid primary key candidate should generally:

- Exist in the table.
- Have no missing values.
- Uniquely identify each row.

Some tables may not have a clear single-column primary key in the raw CSV files. This is normal and will be handled later during database modeling or cleaning.

In [22]:
relationship_candidates = [
    {
        "from_table": "orders",
        "from_column": "customer_id",
        "to_table": "customers",
        "to_column": "customer_id"
    },
    {
        "from_table": "order_items",
        "from_column": "order_id",
        "to_table": "orders",
        "to_column": "order_id"
    },
    {
        "from_table": "order_items",
        "from_column": "product_id",
        "to_table": "products",
        "to_column": "product_id"
    },
    {
        "from_table": "order_items",
        "from_column": "seller_id",
        "to_table": "sellers",
        "to_column": "seller_id"
    },
    {
        "from_table": "order_payments",
        "from_column": "order_id",
        "to_table": "orders",
        "to_column": "order_id"
    },
    {
        "from_table": "order_reviews",
        "from_column": "order_id",
        "to_table": "orders",
        "to_column": "order_id"
    },
    {
        "from_table": "products",
        "from_column": "product_category_name",
        "to_table": "category_translation",
        "to_column": "product_category_name"
    }
]

In [23]:
foreign_key_records = []

for relationship in relationship_candidates:
    from_table = relationship["from_table"]
    from_column = relationship["from_column"]
    to_table = relationship["to_table"]
    to_column = relationship["to_column"]

    from_values = dfs[from_table][from_column].dropna()
    to_values = set(dfs[to_table][to_column].dropna())

    unmatched_values = ~from_values.isin(to_values)

    foreign_key_records.append({
        "from_table": from_table,
        "from_column": from_column,
        "to_table": to_table,
        "to_column": to_column,
        "source_non_null_rows": len(from_values),
        "unmatched_rows": unmatched_values.sum(),
        "unmatched_percentage": unmatched_values.mean() * 100 if len(from_values) > 0 else 0
    })

foreign_key_summary = pd.DataFrame(foreign_key_records)

display(foreign_key_summary)

,from_table,from_column,to_table,to_column,source_non_null_rows,unmatched_rows,unmatched_percentage
0,orders,customer_id,customers,customer_id,99441,0,0.00
1,order_items,order_id,orders,order_id,112650,0,0.00
2,order_items,product_id,products,product_id,112650,0,0.00
3,order_items,seller_id,sellers,seller_id,112650,0,0.00
4,order_payments,order_id,orders,order_id,103886,0,0.00
5,order_reviews,order_id,orders,order_id,99224,0,0.00
6,products,product_category_name,category_translation,product_category_name,32341,13,0.04


### Preliminary Foreign Key Notes

The relationship checks above compare values between likely related tables.

A low or zero unmatched percentage suggests that the relationship is consistent in the raw data. Any unmatched values should be investigated later before building the SQL layer or analytical dataset.

These checks are preliminary and do not modify the raw data.

## 9. Future Modeling Consideration: Data Leakage

The future target for the machine learning phase will be based on whether an order was delivered after its estimated delivery calendar date:

`is_late = delivered_date > estimated_delivery_date`

where:

- `delivered_date` is derived from `order_delivered_customer_date`
- `estimated_delivery_date` is derived from `order_estimated_delivery_date`

This comparison must be made at calendar-date level, not full timestamp level. Orders delivered on the estimated delivery date are not considered late because the estimated delivery field represents a promised delivery date rather than an exact delivery time.

This date-level definition is more appropriate for business interpretation because the estimated delivery field represents a promised delivery date rather than an exact promised delivery time.

However, the predictive model must only use information that would be available before delivery happens.

Therefore, future modeling datasets should not use post-delivery variables such as:

- `order_delivered_customer_date`
- calculated delivery delay fields
- review information such as `review_score`
- final delivery outcome variables
- any feature only known after the delivery has already occurred

The target can be created from post-delivery information for supervised learning, but those same post-delivery fields must not be used as predictive features.

This notebook does not create the target or the final machine learning dataset. This rule is documented here as an important future constraint.

## 10. Dataset Documentation

This section summarizes what each raw table represents, how it may connect to the rest of the dataset, and how it may be useful later in the project.

The goal is to document the raw data structure from a business and analytical perspective before performing any cleaning or transformations.

In [28]:
table_documentation = pd.DataFrame([
    {
        "table_name": "customers",
        "description": "Customer location and customer identifiers.",
        "relationship_notes": "Connects to orders through customer_id.",
        "future_project_use": "Useful for customer geography and regional delivery risk analysis.",
        "leakage_consideration": "Generally safe if used as customer/location information available before delivery."
    },
    {
        "table_name": "geolocation",
        "description": "Brazilian zip code prefixes with latitude, longitude, city, and state information.",
        "relationship_notes": "Can be connected indirectly to customers and sellers through zip code prefixes.",
        "future_project_use": "Useful for geographic enrichment, distance estimation, and regional logistics analysis.",
        "leakage_consideration": "Generally safe, but requires careful aggregation because zip prefixes may appear multiple times."
    },
    {
        "table_name": "orders",
        "description": "Core order table containing order status and key order timestamps.",
        "relationship_notes": "Central table. Connects to customers, order items, payments, and reviews through order_id.",
        "future_project_use": "Main base table for the future delivery delay target and order-level dataset.",
        "leakage_consideration": "Requires strict leakage control. Delivered dates and final delivery status should not be used as model features."
    },
    {
        "table_name": "order_items",
        "description": "Order item-level details, including products, sellers, prices, freight values, and shipping limit dates.",
        "relationship_notes": "Connects orders to products and sellers through order_id, product_id, and seller_id.",
        "future_project_use": "Useful for product, seller, price, freight, and order complexity features.",
        "leakage_consideration": "Mostly useful for pre-delivery features, but timestamp availability should be reviewed later."
    },
    {
        "table_name": "order_payments",
        "description": "Payment information related to each order, including payment type, installments, and payment value.",
        "relationship_notes": "Connects to orders through order_id. Orders may have multiple payment rows.",
        "future_project_use": "Useful for payment behavior and order value features.",
        "leakage_consideration": "Likely safe if payment information is known at purchase time."
    },
    {
        "table_name": "order_reviews",
        "description": "Customer review information, including review scores and comments.",
        "relationship_notes": "Connects to orders through order_id.",
        "future_project_use": "Useful for post-delivery satisfaction analysis, but not for predictive model features.",
        "leakage_consideration": "High leakage risk. Review data is post-delivery and should not be used as model input."
    },
    {
        "table_name": "products",
        "description": "Product attributes such as category, dimensions, weight, photos, and text metadata.",
        "relationship_notes": "Connects to order_items through product_id and to category_translation through product_category_name.",
        "future_project_use": "Useful for product category, size, weight, and complexity features.",
        "leakage_consideration": "Generally safe if product attributes are available before shipment."
    },
    {
        "table_name": "sellers",
        "description": "Seller location and seller identifiers.",
        "relationship_notes": "Connects to order_items through seller_id.",
        "future_project_use": "Useful for seller geography, seller distribution, and logistics route analysis.",
        "leakage_consideration": "Generally safe if used as seller/location information available before delivery."
    },
    {
        "table_name": "category_translation",
        "description": "Translation table for product category names from Portuguese to English.",
        "relationship_notes": "Connects to products through product_category_name.",
        "future_project_use": "Useful for making product categories easier to interpret in analysis, dashboards, and presentation.",
        "leakage_consideration": "Safe reference table."
    }
])
with pd.option_context("display.max_colwidth", None):
    display(table_documentation)

,table_name,description,relationship_notes,future_project_use,leakage_consideration
0,customers,Customer location and customer identifiers.,Connects to orders through customer_id.,Useful for customer geography and regional delivery risk analysis.,Generally safe if used as customer/location information available before delivery.
1,geolocation,"Brazilian zip code prefixes with latitude, longitude, city, and state information.",Can be connected indirectly to customers and sellers through zip code prefixes.,"Useful for geographic enrichment, distance estimation, and regional logistics analysis.","Generally safe, but requires careful aggregation because zip prefixes may appear multiple times."
2,orders,Core order table containing order status and key order timestamps.,"Central table. Connects to customers, order items, payments, and reviews through order_id.",Main base table for the future delivery delay target and order-level dataset.,Requires strict leakage control. Delivered dates and final delivery status should not be used as model features.
3,order_items,"Order item-level details, including products, sellers, prices, freight values, and shipping limit dates.","Connects orders to products and sellers through order_id, product_id, and seller_id.","Useful for product, seller, price, freight, and order complexity features.","Mostly useful for pre-delivery features, but timestamp availability should be reviewed later."
4,order_payments,"Payment information related to each order, including payment type, installments, and payment value.",Connects to orders through order_id. Orders may have multiple payment rows.,Useful for payment behavior and order value features.,Likely safe if payment information is known at purchase time.
5,order_reviews,"Customer review information, including review scores and comments.",Connects to orders through order_id.,"Useful for post-delivery satisfaction analysis, but not for predictive model features.",High leakage risk. Review data is post-delivery and should not be used as model input.
6,products,"Product attributes such as category, dimensions, weight, photos, and text metadata.",Connects to order_items through product_id and to category_translation through product_category_name.,"Useful for product category, size, weight, and complexity features.",Generally safe if product attributes are available before shipment.
7,sellers,Seller location and seller identifiers.,Connects to order_items through seller_id.,"Useful for seller geography, seller distribution, and logistics route analysis.",Generally safe if used as seller/location information available before delivery.
8,category_translation,Translation table for product category names from Portuguese to English.,Connects to products through product_category_name.,"Useful for making product categories easier to interpret in analysis, dashboards, and presentation.",Safe reference table.


## 11. Data Understanding Summary

This final section summarizes the main outputs generated in the data understanding notebook.

No cleaning, feature engineering, target creation, or modeling has been performed in this notebook. The objective was to inspect and document the raw data structure.

In [25]:
tables_with_duplicates = table_summary.loc[table_summary["duplicate_rows"] > 0, "table_name"].tolist()
tables_with_missing = missing_values_summary["table_name"].unique().tolist() if not missing_values_summary.empty else []
relationships_with_unmatched_rows = foreign_key_summary.loc[
    foreign_key_summary["unmatched_rows"] > 0,
    ["from_table", "from_column", "to_table", "to_column", "unmatched_rows", "unmatched_percentage"]
]

notebook_summary = pd.DataFrame([
    {
        "area": "Raw tables loaded",
        "summary": f"{len(dfs)} tables loaded into the dfs dictionary.",
        "status": "Completed"
    },
    {
        "area": "Raw file validation",
        "summary": "All expected Olist CSV files were validated before loading.",
        "status": "Completed"
    },
    {
        "area": "Duplicate rows",
        "summary": f"Tables with duplicate rows: {tables_with_duplicates if tables_with_duplicates else 'None detected'}",
        "status": "Reviewed"
    },
    {
        "area": "Missing values",
        "summary": f"Tables with missing values: {tables_with_missing if tables_with_missing else 'None detected'}",
        "status": "Reviewed"
    },
    {
        "area": "Datetime-like columns",
        "summary": "Potential datetime-like columns were detected but not converted.",
        "status": "Deferred to cleaning"
    },
    {
        "area": "Relational structure",
        "summary": "Preliminary primary key and foreign key checks were created.",
        "status": "Reviewed"
    },
    {
        "area": "Data leakage",
        "summary": "Future leakage risks were documented, especially post-delivery variables.",
        "status": "Documented"
    }
])
with pd.option_context("display.max_colwidth", None):
    display(notebook_summary)

,area,summary,status
0,Raw tables loaded,9 tables loaded into the dfs dictionary.,Completed
1,Raw file validation,All expected Olist CSV files were validated before loading.,Completed
2,Duplicate rows,Tables with duplicate rows: ['geolocation'],Reviewed
3,Missing values,"Tables with missing values: ['order_reviews', 'orders', 'products']",Reviewed
4,Datetime-like columns,Potential datetime-like columns were detected but not converted.,Deferred to cleaning
5,Relational structure,Preliminary primary key and foreign key checks were created.,Reviewed
6,Data leakage,"Future leakage risks were documented, especially post-delivery variables.",Documented


In [26]:
if relationships_with_unmatched_rows.empty:
    print("No unmatched rows detected in the preliminary foreign key checks.")
else:
    display(relationships_with_unmatched_rows)

,from_table,from_column,to_table,to_column,unmatched_rows,unmatched_percentage
6,products,product_category_name,category_translation,product_category_name,13,0.04


## 12. Conclusions

The initial data understanding phase has been completed.

The raw Olist dataset was successfully loaded, validated, and inspected at table level. The notebook documents the structure of the 9 expected raw CSV files, including table sizes, columns, data types, missing values, duplicate rows, possible datetime fields, and preliminary relational links.

Key conclusions from this phase:

- The project should treat `orders` as the central table for future analytical and machine learning datasets.
- The dataset has a relational structure that must be respected when joining tables.
- Some tables may contain multiple rows per order, especially `order_items`, `order_payments`, and potentially `order_reviews`.
- Datetime columns should be converted carefully in the cleaning phase.
- Missing values and duplicate rows should be handled later, not in this notebook.
- Review information should be treated as post-delivery data and excluded from predictive model features.
- The future target will be based on delivery lateness, but the target is not created in this notebook.